# ⭐ Google Maps Review Crawler (hotel)

Chạy lần lượt các cell từ trên xuống: **① Cấu hình → ② Đọc input → ③ Crawl → ④ Xem kết quả**.

**Input:** Google Sheet (cột 1 = link Google Maps của hotel — nhận cả link rút gọn `maps.app.goo.gl/...` lẫn link đầy đủ `google.com/maps/place/...`) hoặc file CSV/XLSX/TXT offline (sửa `INPUT_MODE` ở cell ①).

**Không cần browser:** crawl trực tiếp qua API bằng curl_cffi — kỹ thuật capture & replay giống crawler giá Agoda (xem chi tiết endpoint trong `gmaps_review.py`).

**Output** nằm trong `results/gmaps/<RUN_NAME>/`:
- `FINAL_hotels_<YYYYMMDD>.csv` — 1 dòng/hotel: rating tổng, tổng số đánh giá, địa chỉ…
- `FINAL_reviews_<YYYYMMDD>.csv` — 1 dòng/review: nguồn, người viết, số sao, ngày (tương đối + ISO xấp xỉ), nội dung, phản hồi của khách sạn…
- `TEMP_gmaps_*.csv` — checkpoint: lỡ tắt giữa chừng, chạy lại cell ③ sẽ tự resume — hotel **chưa từng cào** chạy trước, hotel lỗi retry sau.

⚠️ Review trên trang hotel của Google gồm **mọi nguồn** (Google, Tripadvisor, Booking, Trip.com…) — cột `source` cho biết nguồn; chỉ cần review Google thì lọc `source == "Google"`. Thứ tự review là "liên quan nhất" (mặc định của Google), nên khi đặt `MAX_REVIEWS > 0` sẽ lấy N review nổi bật nhất chứ không phải mới nhất.

ℹ️ `review_count` (file hotels) là con số hiển thị trên placecard Google Maps, còn `fetched_reviews` là tổng review thực lấy được từ feed đa nguồn — thường **nhiều hơn** `review_count` là bình thường.

In [1]:
# ════════════════ ① CẤU HÌNH ════════════════

# ── Tên run: kết quả nằm RIÊNG trong results/gmaps/<RUN_NAME>/ ──
RUN_NAME = "run1"

# ── Nguồn input: "gsheet" (online) hoặc "offline" (file trên máy) ──
INPUT_MODE = "gsheet"

# Dùng khi INPUT_MODE = "gsheet" (gid của tab được tự lấy từ URL)
GSHEET_URL = "https://docs.google.com/spreadsheets/d/1Q2DwDDOZKARLymoCU0n_G0ViTx4oXnqIwD09PydAVQU/edit?gid=0#gid=0"

# Dùng khi INPUT_MODE = "offline" — đường dẫn tuyệt đối, hoặc tương đối so với 31.crawl-tool
# File chỉ cần cột 1 là link Google Maps (cột 2 nếu có = tên tuỳ chọn)
OFFLINE_FILE = "input/gmaps_hotels.csv"

# ── Tham số crawl ──
MAX_REVIEWS = 0     # số review tối đa mỗi hotel; 0 = lấy HẾT
MAX_HOTELS  = 0     # 0 = crawl tất cả; đặt 2 để test nhanh 2 hotel đầu
HL          = "vi"  # "vi": nội dung + ngày tiếng Việt (Google tự dịch); "en": tiếng Anh

In [2]:
# ════════════════ ② ĐỌC INPUT ════════════════
import importlib
import os, sys

if "ROOT" not in globals():                    # giữ nguyên ROOT khi chạy lại cell
    NB_DIR = os.path.abspath("")
    ROOT = NB_DIR if os.path.isdir(os.path.join(NB_DIR, "crawler")) else os.path.dirname(NB_DIR)
assert os.path.isdir(os.path.join(ROOT, "crawler")), (
    f"Không tìm thấy package `crawler` quanh {ROOT} — hãy mở notebook từ 31.crawl-tool/google-review")
for p in (ROOT, os.path.join(ROOT, "google-review")):
    if p not in sys.path:
        sys.path.insert(0, p)

import gmaps_review
importlib.reload(gmaps_review)                 # nhận thay đổi nếu vừa sửa gmaps_review.py

if INPUT_MODE == "gsheet":
    INPUT = GSHEET_URL
    print("📡 Input: Google Sheet online")
else:
    INPUT = OFFLINE_FILE if os.path.isabs(OFFLINE_FILE) else os.path.join(ROOT, OFFLINE_FILE)
    assert os.path.exists(INPUT), f"Không tìm thấy file: {INPUT}"
    print(f"📁 Input: file offline — {INPUT}")

links = gmaps_review.read_links(INPUT)
print(f"✅ Đọc được {len(links)} hotel. 5 dòng đầu:")
for url, name in links[:5]:
    print(f"   • {name or '(tên tự lấy từ Google)'} — {url}")

📡 Input: Google Sheet online
✅ Đọc được 2 hotel. 5 dòng đầu:
   • (tên tự lấy từ Google) — https://maps.app.goo.gl/YPArGf22bQ1QsJ756
   • (tên tự lấy từ Google) — https://maps.app.goo.gl/GbcDvc6tGtmhtD8aA


In [3]:
# (TÙY CHỌN) Tải Google Sheet về file offline — lần sau chỉ cần đổi INPUT_MODE = "offline"
import pandas as pd
from crawler.hotels_io import _gsheet_url

os.makedirs(os.path.join(ROOT, "input"), exist_ok=True)
dest = os.path.join(ROOT, "input", "gmaps_hotels.csv")
pd.read_csv(_gsheet_url(GSHEET_URL)).to_csv(dest, index=False, encoding="utf-8-sig")
print(f"💾 Đã lưu bản offline: {dest}")

💾 Đã lưu bản offline: /Users/hchinhtrung/Documents/GitHub/mvillage-email-template/31.crawl-tool/input/gmaps_hotels.csv


In [4]:
# ════════════════ ③ CRAWL ════════════════
OUTDIR = os.path.join(ROOT, "results", "gmaps", RUN_NAME)   # mỗi notebook 1 thư mục riêng

gmaps_review.crawl(
    INPUT,
    out_dir=OUTDIR,
    max_reviews=MAX_REVIEWS,
    max_hotels=MAX_HOTELS,
    hl=HL,
)

🚀 2 hotel trong input | crawl 2 (mới 2, retry 0) | max_reviews=ALL | hl=vi

🏨 1/2 M Village Hotel Tao Đàn Park, a brand of Modern Village Li
   ⭐ 4.8 | 568 đánh giá | 14 Trương Định, Xuân Hòa, Hồ Chí Minh 70000, Việt Nam
   … đã lấy 200 review
   … đã lấy 400 review
   … đã lấy 600 review
   … đã lấy 800 review
   … đã lấy 1000 review
   … đã lấy 1200 review
   ✅ lấy được 1351 review

🏨 2/2 Grand Signature by M Village Lê Thánh Tôn, a brand of Mode
   ⭐ 4.9 | 1482 đánh giá | 24 Lê Thánh Tôn, Sài Gòn, Hồ Chí Minh 700000, Việt Nam
   … đã lấy 200 review
   … đã lấy 400 review
   … đã lấy 600 review
   … đã lấy 800 review
   … đã lấy 1000 review
   … đã lấy 1200 review
   … đã lấy 1400 review
   … đã lấy 1600 review
   … đã lấy 1800 review
   … đã lấy 2000 review
   … đã lấy 2200 review
   … đã lấy 2400 review
   ✅ lấy được 2431 review

🏁 Xong 2/2 hotel, 3782 review trong 485s
📄 /Users/hchinhtrung/Documents/GitHub/mvillage-email-template/31.crawl-tool/results/gmaps/run1/FINAL_hotels_20260

('/Users/hchinhtrung/Documents/GitHub/mvillage-email-template/31.crawl-tool/results/gmaps/run1/FINAL_hotels_20260716.csv',
 '/Users/hchinhtrung/Documents/GitHub/mvillage-email-template/31.crawl-tool/results/gmaps/run1/FINAL_reviews_20260716.csv')

In [5]:
# ════════════════ ④ XEM KẾT QUẢ ════════════════
import glob
import pandas as pd

OUTDIR = os.path.join(ROOT, "results", "gmaps", RUN_NAME)
fh = sorted(glob.glob(os.path.join(OUTDIR, "FINAL_hotels_*.csv")))
fr = sorted(glob.glob(os.path.join(OUTDIR, "FINAL_reviews_*.csv")))
assert fh and fr, "Chưa có file FINAL nào — hãy chạy cell ③ trước."

hotels = pd.read_csv(fh[-1])
reviews = pd.read_csv(fr[-1])
print(f"📄 {fh[-1]} — {len(hotels)} hotel")
print(f"📄 {fr[-1]} — {len(reviews)} review "
      f"(Google: {(reviews['source'] == 'Google').sum()}, "
      f"nguồn khác: {(reviews['source'] != 'Google').sum()})")
display(hotels[["hotel_name", "overall_rating", "review_count", "fetched_reviews", "status", "address"]])
reviews[["hotel_name", "source", "author", "rating", "when", "approx_date", "text"]].head(15)

📄 /Users/hchinhtrung/Documents/GitHub/mvillage-email-template/31.crawl-tool/results/gmaps/run1/FINAL_hotels_20260716.csv — 2 hotel
📄 /Users/hchinhtrung/Documents/GitHub/mvillage-email-template/31.crawl-tool/results/gmaps/run1/FINAL_reviews_20260716.csv — 3782 review (Google: 2753, nguồn khác: 1029)


,hotel_name,overall_rating,review_count,fetched_reviews,status,address
0,"M Village Hotel Tao Đàn Park, a brand of Moder...",4.8,568,1351,ok,"14 Trương Định, Xuân Hòa, Hồ Chí Minh 70000, V..."
1,"Grand Signature by M Village Lê Thánh Tôn, a b...",4.9,1482,2431,ok,"24 Lê Thánh Tôn, Sài Gòn, Hồ Chí Minh 700000, ..."


,hotel_name,source,author,rating,when,approx_date,text
0,"M Village Hotel Tao Đàn Park, a brand of Moder...",Tripadvisor,Road820227,4.0,9 tháng trước,2025-10-15,"Phòng nhỏ, phù hợp khách đi công tác. Có khu g..."
1,"M Village Hotel Tao Đàn Park, a brand of Moder...",Google,đức thịnh từ,5.0,5 tháng trước,2026-02-14,"Nhân viên hỗ trợ rất rất nhiệt tình, tiện ích ..."
2,"M Village Hotel Tao Đàn Park, a brand of Moder...",Google,Nguyễn Duy Hưng (Phệ),5.0,2 tháng trước,2026-05-16,"Mọi thứ đều gần như tuyệt vời, có bị 1 vài điể..."
3,"M Village Hotel Tao Đàn Park, a brand of Moder...",Google,Thu Trang Vũ,5.0,2 tháng trước,2026-05-16,"Dịch vụ chu đáo, phòng Ks thiết kế đẹp, màu sắ..."
4,"M Village Hotel Tao Đàn Park, a brand of Moder...",Google,Ha Nguyen,5.0,2 tháng trước,2026-05-16,"Phòng sạch sẽ gọn gàng, kèm câc tiện ích. Các ..."
5,"M Village Hotel Tao Đàn Park, a brand of Moder...",Google,Tạ Hương Linh,5.0,9 tháng trước,2025-10-15,"Sạch sẽ, nhân viên nhiệt tình. Có phòng giặt s..."
6,"M Village Hotel Tao Đàn Park, a brand of Moder...",Google,Ngoc Ngan Nguyen,5.0,5 tháng trước,2026-02-14,"5 sao cho dịch vụ chỉnh chu, phòng ốc sạch sẽ ..."
7,"M Village Hotel Tao Đàn Park, a brand of Moder...",Google,Minh PhanThiCam,5.0,2 tuần trước,2026-07-02,Nhân viên rất nhiệt tình và chuyên nghiệp- hỗ ...
8,"M Village Hotel Tao Đàn Park, a brand of Moder...",Google,huyen Trang,5.0,5 tháng trước,2026-02-14,Mọi thứ rất ok. Mình sẽ ghé thêm nhiều lần nữa...
9,"M Village Hotel Tao Đàn Park, a brand of Moder...",Google,ĐINH HIỆP,5.0,một tháng trước,2026-06-16,1 địa điểm đáng để lựa chọn đối với khách du l...
